# Read my handwriting (Google Colab)

Fine-tunes TrOCR on handwritten maths and exports it for the `TrOCROCR` adapter,
which sits beside `TesseractOCR` behind the same `OCRProvider` protocol — so the
vault pipeline picks it up with no other change.

**Why not just Tesseract:** it was built for printed text. On a photographed
exercise book it will happily return `2x + |` for `2x + 1` and `S` for `5`, and
the solver then answers a question nobody asked. Handwriting is a different
problem, not a harder setting.

**Runtime:** *Runtime ▸ Change runtime type ▸ GPU*.


In [ ]:
!pip install -q transformers datasets jiwer evaluate optimum[onnxruntime] pillow
import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 1. The data

Two options, and they are not equivalent:

* **Your own pages** — photograph 200-500 lines of your handwriting and type the
  transcript for each. Tedious, and by far the best result: it learns *your*
  sevens.
* **A public set** (IAM, CROHME) — free, and teaches it someone else's hand.

Layout either way: `data/images/*.png` and `data/labels.csv` with `file,text`.

In [ ]:
from google.colab import files
import zipfile, os, glob, pandas as pd
os.makedirs('data', exist_ok=True)
up = files.upload()
for name in up:
    if name.endswith('.zip'):
        zipfile.ZipFile(name).extractall('data')

labels = glob.glob('data/**/labels.csv', recursive=True)
assert labels, 'no labels.csv found'
df = pd.read_csv(labels[0])
root = os.path.dirname(labels[0])
df['path'] = df['file'].apply(lambda f: os.path.join(root, 'images', f))
missing = [p for p in df['path'] if not os.path.exists(p)]
assert not missing, f'{len(missing)} labelled images are missing, e.g. {missing[:3]}'
print(len(df), 'labelled lines')
df.head()

## 2. Dataset

In [ ]:
from torch.utils.data import Dataset
from transformers import TrOCRProcessor
from PIL import Image

BASE = 'microsoft/trocr-base-handwritten'
processor = TrOCRProcessor.from_pretrained(BASE)

class Lines(Dataset):
    def __init__(self, frame): self.frame = frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        image = Image.open(row['path']).convert('RGB')
        pixel_values = processor(image, return_tensors='pt').pixel_values[0]
        labels = processor.tokenizer(
            str(row['text']), padding='max_length', max_length=64, truncation=True,
        ).input_ids
        # -100 is ignored by the loss; padding must not be learned as content.
        labels = [t if t != processor.tokenizer.pad_token_id else -100 for t in labels]
        return {'pixel_values': pixel_values, 'labels': torch.tensor(labels)}

split = int(len(df) * 0.9)
train_ds, eval_ds = Lines(df[:split]), Lines(df[split:])
print(len(train_ds), 'train |', len(eval_ds), 'eval')

## 3. Fine-tune

In [ ]:
from transformers import VisionEncoderDecoderModel, Seq2SeqTrainer, Seq2SeqTrainingArguments
import evaluate, numpy as np

model = VisionEncoderDecoderModel.from_pretrained(BASE)
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id

cer = evaluate.load('cer')
def metrics(pred):
    ids = pred.label_ids.copy()
    ids[ids == -100] = processor.tokenizer.pad_token_id
    return {'cer': cer.compute(
        predictions=processor.batch_decode(pred.predictions, skip_special_tokens=True),
        references=processor.batch_decode(ids, skip_special_tokens=True))}

trainer = Seq2SeqTrainer(
    model=model,
    args=Seq2SeqTrainingArguments(
        output_dir='trocr-friday', predict_with_generate=True,
        per_device_train_batch_size=4, per_device_eval_batch_size=4,
        num_train_epochs=8, fp16=torch.cuda.is_available(),
        eval_strategy='epoch', save_strategy='epoch', logging_steps=25,
        load_best_model_at_end=True, metric_for_best_model='cer', greater_is_better=False),
    train_dataset=train_ds, eval_dataset=eval_ds,
    compute_metrics=metrics, tokenizer=processor.feature_extractor,
)
trainer.train()

## 4. Is it actually better?

Character error rate against the same held-out lines, fine-tuned versus stock.
If it has not beaten the base model, do not ship it — a worse OCR feeding a
confident solver is the bad outcome this whole track exists to avoid.

In [ ]:
base = VisionEncoderDecoderModel.from_pretrained(BASE).to(model.device)
base.config.decoder_start_token_id = processor.tokenizer.cls_token_id
base.config.pad_token_id = processor.tokenizer.pad_token_id

def cer_of(m):
    preds, refs = [], []
    for i in range(len(eval_ds)):
        item = eval_ds[i]
        out = m.generate(item['pixel_values'].unsqueeze(0).to(m.device), max_length=64)
        preds.append(processor.batch_decode(out, skip_special_tokens=True)[0])
        ids = item['labels'].clone(); ids[ids == -100] = processor.tokenizer.pad_token_id
        refs.append(processor.decode(ids, skip_special_tokens=True))
    return cer.compute(predictions=preds, references=refs)

stock, tuned = cer_of(base), cer_of(model)
print('stock TrOCR CER : %.4f' % stock)
print('fine-tuned CER  : %.4f' % tuned)
print('VERDICT:', 'ship it' if tuned < stock else 'DO NOT SHIP - it got worse')

## 5. Export

ONNX, so the adapter needs `optimum`/`onnxruntime` rather than the whole
training stack at inference time.

In [ ]:
from optimum.onnxruntime import ORTModelForVision2Seq
model.save_pretrained('trocr-friday/final'); processor.save_pretrained('trocr-friday/final')
ORTModelForVision2Seq.from_pretrained('trocr-friday/final', export=True).save_pretrained('trocr-onnx')
processor.save_pretrained('trocr-onnx')
!du -sh trocr-onnx && zip -qr trocr-onnx.zip trocr-onnx
from google.colab import files; files.download('trocr-onnx.zip')

## 6. Install it

Unzip to `models/ocr/trocr/`, then:

```bash
FRIDAY_OCR_PROVIDER=trocr
FRIDAY_TROCR_MODEL=models/ocr/trocr
```

`TrOCROCR` in `src/friday/perception/ocr.py` imports `optimum` lazily, so an
install without the extras is unaffected until the provider is actually
selected.
